# Libraries

In [1]:
import torch
from transformers import MarianMTModel, MarianTokenizer
import pandas as pd
import random, re
from tqdm import tqdm

tqdm.pandas()

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# Load Data

In [2]:
df = pd.read_csv('/content/MITweet_Auth_Lib.csv')

In [3]:
df.head()

,tweet,label
0,Forget about protecting the people and communi...,1
1,Are you with an #environmentalnonprofit leadin...,0
2,Curious if #IRA can help you electrify your ho...,1
3,Learn more from \n on how huge investments fro...,0
4,Almost a year since this Vox video on grid cha...,1


# Data Construction

## Translating to Indonesian

leaving about 10-30% of the words still in english

In [4]:
model_name = "Helsinki-NLP/opus-mt-en-id"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [5]:
def translate_en_to_id(text):
    """Translate English -> Indonesian using CUDA."""
    try:
        encoded = tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            output = model.generate(
                **encoded,
                max_length=128,
                num_beams=4
            )

        return tokenizer.decode(output[0], skip_special_tokens=True)

    except:
        return str(text)


In [6]:
def translate_en_to_id_batch(text_list):
    translated = []
    for t in text_list:
        translated.append(translate_en_to_id(t))
    return translated

## Inject Slang words

In [7]:
slang_dict = {
    # Pronouns / People
    "saya": "gw", "aku": "gue", "kami": "kita", "kamu": "lu", "anda": "lu",
    "dia": "dia", "mereka": "mereka", "teman": "temen", "kawan": "sob",
    "saudara": "bro", "istri": "bini", "suami": "laki", "ayah": "bokap",
    "ibu": "nyokap", "anak": "bocah", "orang": "org",

    # Verbs
    "pergi": "cabut", "datang": "dateng", "makan": "makan", "minum": "ngopi",
    "berbicara": "ngomong", "bicara": "ngomong", "melihat": "ngeliat",
    "menunggu": "nunggu", "membaca": "baca", "menulis": "nulis",
    "belajar": "belajar", "mengerti": "ngerti", "paham": "ngeh",
    "membeli": "beli", "memakai": "make", "mengambil": "ngambil",
    "mengirim": "ngirim", "menjawab": "bales", "memberikan": "ngasih",
    "mengerjakan": "ngerjain",

    # Negations / Modals
    "tidak": "ga", "nggak": "ga", "bukan": "bukannya", "akan": "bakal",
    "ingin": "pengen", "sudah": "udah", "belum": "blm", "harus": "musti",
    "dapat": "bisa", "mungkin": "mungkin aja", "kenapa": "ngapa",
    "bagaimana": "gimana", "apa": "apaan",

    # Expressions
    "sekali": "banget", "sangat": "super", "benar": "bener",
    "asal": "aslinya", "ternyata": "ternyata loh", "serius": "seriusan",
    "parah": "parah banget", "lelah": "capek", "menyedihkan": "sedih banget",
    "kecewa": "ilfil", "marah": "kesel", "bodoh": "bege", "jelek": "ampas",
    "bagus": "keren", "hebat": "mantap", "lucu": "ngakak",

    # Chat Slang
    "tertawa": "wkwk", "ketawa": "wkwkwk", "lol": "wkwk", "sungguh": "anjir",
    "astaga": "anjay", "wow": "gila sih", "bodoh sekali": "goblok banget",

    # Shortenings
    "yang": "yg", "dan": "dn", "kamu": "km", "sudah": "sdh",
    "tidak": "tdk", "terima kasih": "makasih", "tolong": "pls",
    "banget": "bgt", "sekali": "scl", "ayo": "yuk",

    # Places
    "rumah": "rumah", "sekolah": "skul", "kampus": "kampus",
    "kantor": "kantor", "tempat kerja": "workplace", "jalan": "jalanan",
    "mobil": "mobe", "motor": "montor", "kota": "kota", "desa": "kampung",

    # Gaul
    "iya": "iyaa", "enggak": "ga", "maksudnya": "mksdnya",
    "gila": "gilaaa", "keren": "keren sumpah", "mantap": "mantul",
    "oke": "oke lah", "bro": "cuy", "maaf": "sorry",

    # Politics / Social Commentary
    "politik": "perpolan", "pemerintah": "pemerintah",
    "kebijakan": "kbijakan", "kampanye": "kampanye",
    "pemilu": "pemilu", "isu": "isu", "berita": "news", "media": "medsos",
    "partai": "partai", "korupsi": "korup", "demokrasi": "demo",
    "kritik": "nyinyir", "menolak": "nolakin"
}

In [8]:
def inject_slang(text, slang_dict, probability=0.5):
    words = text.split()
    result = []

    for w in words:
        lw = w.lower()

        if lw in slang_dict and random.random() < probability:
            result.append(slang_dict[lw])
        else:
            result.append(w)

    return " ".join(result)


## Code Switching

In [9]:
def add_code_switch(original_list, indo_slang_list, keep_ratio=0.2):
    final = []

    for original, slang in zip(original_list, indo_slang_list):
        eng_tokens = original.split()

        if len(eng_tokens) == 0:
            final.append(slang)
            continue

        num_keep = max(1, int(len(eng_tokens) * keep_ratio))
        keep_words = random.sample(eng_tokens, num_keep)

        indo_tokens = slang.split()

        # prevent crash here
        if len(indo_tokens) == 0:
            final.append(slang)
            continue

        for ew in keep_words:
            idx = random.randint(0, len(indo_tokens) - 1)
            indo_tokens.insert(idx, ew)

        final.append(" ".join(indo_tokens))

    return final


## Apply All

In [10]:
def process_all(df, batch_size=32):
    results = []

    for i in tqdm(range(0, len(df), batch_size)):
        chunk = df["tweet"].iloc[i : i + batch_size].tolist()

        # Step 1 — Translate English → Indonesian
        translated = translate_en_to_id_batch(chunk)

        # Step 2 — Slangify
        slangified = [inject_slang(t, slang_dict) for t in translated]

        # Step 3 — Add code-switching
        final_batch = add_code_switch(chunk, slangified, keep_ratio=0.25)

        results.extend(final_batch)

    return results


## Test 10 Samples

In [11]:
test_df = df.head(10).copy()
translated_batch = process_all(test_df, batch_size=10)

test_df["tweet_translated"] = translated_batch
test_df[["tweet", "tweet_translated"]]

100%|██████████| 1/1 [00:04<00:00,  4.34s/it]


,tweet,tweet_translated
0,Forget about protecting the people and communi...,Lupakan our tentang melindungi orang-orang han...
1,Are you with an #environmentalnonprofit leadin...,the Apakah ahead: lu on dengan #environmentaln...
2,Curious if #IRA can help you electrify your ho...,Penasaran apakah #IRA bisa help a house? memba...
3,Learn more from \n on how huge investments fro...,Pelajari lebih lanjut dari pada bagaimana besa...
4,Almost a year since this Vox video on grid cha...,since Hampir on setahun planning sejak video V...
5,Whether it’s a downturn in the market or plann...,Apakah itu penurunan pasar #PersonalFinance of...
6,Couldn’t agree more with ⁦\n⁩ on last weeks po...,⁦ education? apaan gunanya mempersiapkan infra...
7,Read this op-ed on the passage of the #IRA fro...,"Baca ini op-ed easy many family."" pada bagian ..."
8,"With the fall of Roe, you need to know who you...","dengan jatuhnya Roe, km perlu tahu siapa yg km..."
9,"NEW CLINIC TARGETED - THE CORNER CLINIC, DUNDE...",Kami menerima Dundee konfirmasi pagi w ini bah...


In [12]:
for i in range(10):
    print("Original:", test_df.loc[i, "tweet"])
    print("Translated:", test_df.loc[i, "tweet_translated"])
    print()


Original: Forget about protecting the people and communities like the ones Cuellar is supposed to represent. 

#IRA doesn't mandate lower emissions. It's about once again giving oil and gas industries a handout so they can continue to pollute our air, water, and land. 

#greenwashing
Translated: Lupakan our tentang melindungi orang-orang handout dn komunitas seperti air, yang Cuellar seharusnya a mewakili. #IRA tidak mewajibkan emisi yg lebih rendah. ini tentang protecting sekali lagi Cuellar memberikan water, minyak dan industri gas sebuah they handout mandate sehingga mereka oil bisa terus mencemari udara, air, dn tanah kita. #greenwashing

Original: Are you with an #environmentalnonprofit leading the way on #IRA issues? Join 
 and grow your support for the important work ahead: #InflationReductionAct
Translated: the Apakah ahead: lu on dengan #environmentalnonprofit yang memimpin isu #IRA? Gabung the dn tingkatkan with dukungan lu untuk pekerjaan penting di depan: #InflasiReduction 

In [13]:
df["tweet_translated"] = process_all(df, batch_size=15)

100%|██████████| 338/338 [29:11<00:00,  5.18s/it]


In [14]:
df.head()

,tweet,label,tweet_translated
0,Forget about protecting the people and communi...,1,they Lupakan supposed continue tentang melindu...
1,Are you with an #environmentalnonprofit leadin...,0,Apakah lu dengan and #environmentalnonprofit w...
2,Curious if #IRA can help you electrify your ho...,1,Penasaran apakah #IRA bisa handy INCREDIBLE me...
3,Learn more from \n on how huge investments fro...,0,& Pelajari lebih transitions lanjut dari pada ...
4,Almost a year since this Vox video on grid cha...,1,Hampir setahun sejak Vox curtain video Vox di ...


In [15]:
df.to_csv("MITweet_Translated_Slang.csv", index=False)